# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dv-06/flyrank-ml_1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule is to prioritize webpages that have high impressions but show signs of declining performance, such as low CTR or poor average position. These pages are more likely to benefit from a content refresh.

Reason codes:
- High impressions
- Low CTR
- Poor average position
- Older content

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!git clone https://github.com/dv-06/flyrank-ml_1.git
%cd flyrank-ml_1
import pandas as pd


df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

print(df[features].head())

Cloning into 'flyrank-ml_1'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 216 (delta 99), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (216/216), 1.96 MiB | 8.47 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/flyrank-ml_1/flyrank-ml_1
   impressions_90d   ctr  avg_position  content_age_days
0             3803  0.76          10.6               187
1            15320  0.05          20.3               445
2            12581  0.09          36.5               141
3            11751  0.49           6.2               463
4            19140  0.13          44.0               263


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The rule assigns a score based on search visibility and performance metrics. Pages with higher scores are placed at the top of the ranked queue so editors can review them first.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Simple baseline rule
score = (
    (df["impressions_90d"] / df["impressions_90d"].max()) * 40
    + ((1 - df["ctr"]) * 30)
    + (df["avg_position"] / df["avg_position"].max()) * 20
    + (df["content_age_days"] / df["content_age_days"].max()) * 10
)

df["baseline_score"] = score

df = df.sort_values("baseline_score", ascending=False)

os.makedirs("../../work/outputs", exist_ok=True)

df.to_csv("../../work/outputs/baseline_action_score.csv", index=False)

df[[
    "baseline_score",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]].head(20)


,baseline_score,impressions_90d,ctr,avg_position,content_age_days
6653,75.664134,517715,0.14,4.2,537
26844,72.940279,509252,0.15,2.5,445
17812,70.784066,517109,0.25,5.4,445
19636,69.980686,497727,0.10,22.2,153
29400,65.539827,443434,0.21,27.9,299
29879,64.127773,416180,0.23,4.0,482
21819,61.558362,463103,0.41,2.3,445
18870,58.695056,345111,0.21,5.4,445
24445,55.514262,1,0.00,245.0,311
7678,55.278843,272144,0.03,2.3,280


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top-ranked pages to make sure the recommendations matched the rule. Pages with high impressions and weaker performance appeared near the top, which is consistent with the goal of prioritizing pages for content refresh. These recommendations should still be checked by an editor before making changes.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = df.head(20)

print(top20[[
    "baseline_score",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]])



       baseline_score  impressions_90d   ctr  avg_position  content_age_days
6653        75.664134           517715  0.14           4.2               537
26844       72.940279           509252  0.15           2.5               445
17812       70.784066           517109  0.25           5.4               445
19636       69.980686           497727  0.10          22.2               153
29400       65.539827           443434  0.21          27.9               299
29879       64.127773           416180  0.23           4.0               482
21819       61.558362           463103  0.41           2.3               445
18870       58.695056           345111  0.21           5.4               445
24445       55.514262                1  0.00         245.0               311
7678        55.278843           272144  0.03           2.3               280
26798       55.195617           286608  0.06          26.2               153
15405       54.979495           214047  0.01          85.8                98

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some recommendations may not actually need a content refresh because performance can change for reasons that are not included in the dataset. I also checked that no label-derived fields, future information, or product decision flags were used when creating the ranking.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = ["trend_pct"]

for feature in excluded:
    if feature in df.columns:
        print(feature, "excluded to avoid leakage")
    else:
        print(feature, "not present")


trend_pct excluded to avoid leakage


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.